## Implementation of a simple form of LLM
#### (without any additional enhancement on query / retrieved docs / routing)

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph

import os, re
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - Basic LLM Testing"

# Initialize LLM
REASONING = True

llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=True if REASONING else False)
emb = OllamaEmbeddings(model="bge-m3:567m")

In [2]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

# Initialize retriever for queries
client = Client(Settings())
client = chromadb.PersistentClient(path="../chroma_db")

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_57713/2343322922.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


In [3]:
# Defining the class structure for the LLM
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# The LLM prompt
LLM_prompt = \
    """
    You are an professional academic advisor in The Hong Kong Polytechnic University, please adhere to the following rules:
        1. Answer in the same language as the user query, e.g. English query, English answer.
        2. Be affirm, avoid saying "may", "maybe", or anything similar,
        3. Say no if you cannot answer the question, do not fabricate factually-false answer,
        4. Provide advice to the student if necessary.

    Now, please use the following context to answer the student's question.
    Remember to thank the user at the end and ask if there are any more enquiry.

    *Context*:
    {context}

    *Student's Question*:
    {question}

    Helpful Answer:
    """
prompt = PromptTemplate.from_template(LLM_prompt)

# Functions for document retrieval based on cos-sim
def retrieve(state: State):
    retrieved_docs = vectorStore.similarity_search(state["question"], k=5)
    return {"context": retrieved_docs}

# Functions for constructing the final LLM prompt
def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    #print(response.additional_kwargs)

    # Include the reasoning part in the output
    return {"answer": f"<think>\n{response.additional_kwargs.get("reasoning_content", "")}</think>\n\n{response.content}"}

    # NOT Include the reasoning part
    #return {"answer": f"{response.content}"}

# Functions for graph building (a process sequence)
def graph_building():
    global graph
    graph_builder = StateGraph(State).add_sequence([retrieve, generate])
    graph_builder.add_edge(START, "retrieve")
    graph = graph_builder.compile()

graph_building()

In [ ]:
query = \
"What is the general university requirement (GUR) for studying in PolyU?"

dataset = []
print(f"Generating {query}")
result = graph.invoke({"question": query})
dataset.append(
    {
        "user_input": result['question'],
        "retrieved_contexts": [doc.page_content for doc in result['context']],  # Extract text from Documents
        "response": result['answer'],
    }
)
print(f"\nAnswer generated:\n{result['answer']}")

Generating What is the general university requirement for studying in PolyU?

Answer generated:
<think>
Hmm, the user is asking about the general university requirements for studying at PolyU. Let me think about what information I have in the context provided.

Looking through the context, I can see details about academic records, GPA requirements, credit requirements, and the selection mechanism for secondary majors. However, the specific information about "general university requirements" isn't explicitly stated here.

The context mentions 27 academic credits for "General University Requirements," but this seems to be part of a program structure rather than the general admission requirements. The GPA requirements mentioned (2.70 for Secondary Major consideration and 1.70 for academic probation) are program-related guidelines, not general university requirements.

Since the question specifically asks about general university requirements, and this information isn't available in the pr